## Install dependencies

In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
!pip install opencv-python-headless

In [ ]:
pip install mediapipe opencv-python matplotlib



## Upload your face image


In [ ]:
from google.colab import files
from PIL import Image
import io
import numpy as np
import cv2
import matplotlib.pyplot as plt

uploaded = files.upload()

if not uploaded:
    raise ValueError("Upload a face image to continue.")

IMAGE_NAME = next(iter(uploaded))
image_rgb = np.array(Image.open(io.BytesIO(uploaded[IMAGE_NAME])).convert("RGB"))

print("Image:", IMAGE_NAME)
print("Image shape:", image_rgb.shape)

plt.figure(figsize=(5, 5))
plt.imshow(image_rgb)
plt.title("Input image")
plt.axis("off")
plt.show()

## Load the three models

In [ ]:
import os
import sys
import pickle
import subprocess
import gdown
import torch
import torch.nn as nn
import torchvision.models as tv_models
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Model 1: custom CNN — random initialisation, not trained
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.features(x)

torch.manual_seed(42)
model_custom = CustomCNN().to(DEVICE).eval()

# Model 2: ImageNet-pretrained ResNet-50
weights = tv_models.ResNet50_Weights.DEFAULT
model_imagenet = tv_models.resnet50(weights=weights)
model_imagenet.fc = nn.Identity()
model_imagenet = model_imagenet.to(DEVICE).eval()

# Model 3: ResNet-50 fine-tuned on VGGFace2
REPO_DIR = "/content/VGGFace2-pytorch"
CHECKPOINT = "/content/resnet50_ft_weight.pkl"
CHECKPOINT_ID = "1A94PAAnwk6L7hXdBXLFosB_s0SzEhAFU"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/cydonia999/VGGFace2-pytorch.git",
         REPO_DIR],
        check=True
    )

if not os.path.isfile(CHECKPOINT) or os.path.getsize(CHECKPOINT) == 0:
    if os.path.exists(CHECKPOINT):
        os.remove(CHECKPOINT)
    gdown.download(id=CHECKPOINT_ID, output=CHECKPOINT, quiet=False)

if not os.path.isfile(CHECKPOINT) or os.path.getsize(CHECKPOINT) == 0:
    raise RuntimeError(
        "VGGFace2 checkpoint download failed. Do not label a different "
        "model as VGGFace2. Obtain a verified checkpoint before continuing."
    )

sys.path.insert(0, REPO_DIR)
from models.resnet import resnet50 as vggface_resnet50

with open(CHECKPOINT, "rb") as f:
    checkpoint_data = pickle.load(f, encoding="latin1")

if isinstance(checkpoint_data, dict):
    if "state_dict" in checkpoint_data:
        checkpoint_data = checkpoint_data["state_dict"]
    elif "model" in checkpoint_data:
        checkpoint_data = checkpoint_data["model"]

state_dict = {}
for key, value in checkpoint_data.items():
    key = key.replace("module.", "")
    if isinstance(value, np.ndarray):
        value = torch.from_numpy(value)
    if torch.is_tensor(value):
        state_dict[key] = value

model_vggface2 = vggface_resnet50(num_classes=8631, include_top=True)
model_vggface2.load_state_dict(state_dict, strict=True)
model_vggface2 = model_vggface2.to(DEVICE).eval()

print("All three models loaded.")
print("IMPORTANT: Custom CNN is untrained.")

## Prepare the five facial regions

In [ ]:
!pip install mediapipe

In [ ]:
import os
import sys
import urllib.request
import bz2
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Download and extract the dlib 68-landmark predictor if not present
PREDICTOR_URL = "http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2"
PREDICTOR_BZ2 = "shape_predictor_68_face_landmarks.dat.bz2"
PREDICTOR_PATH = "shape_predictor_68_face_landmarks.dat"

if not os.path.exists(PREDICTOR_PATH):
    print("Downloading dlib shape predictor (this may take a minute)...")
    urllib.request.urlretrieve(PREDICTOR_URL, PREDICTOR_BZ2)
    print("Decompressing model file...")
    with bz2.BZ2File(PREDICTOR_BZ2, 'rb') as source, open(PREDICTOR_PATH, 'wb') as dest:
        dest.write(source.read())
    os.remove(PREDICTOR_BZ2)
    print("Model ready!")

# 2. Ensure dlib is installed
try:
    import dlib
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "dlib"], check=True)
    import dlib

# 3. Initialize dlib's face detector and shape predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(PREDICTOR_PATH)

# Convert image for dlib processing
gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
rects = detector(gray, 1)

if len(rects) == 0:
    raise RuntimeError("No face detected by dlib. Please try a clearer front-facing image.")

# Get landmarks for the first detected face
shape = predictor(gray, rects[0])
landmarks = np.array([[p.x, p.y] for p in shape.parts()])

# Helper function to dynamically crop regions with padding based on detected landmarks
def get_roi_crop(image, points, padding_ratio=0.2):
    x_min, y_min = np.min(points, axis=0)
    x_max, y_max = np.max(points, axis=0)

    w = x_max - x_min
    h = y_max - y_min

    pad_x = int(w * padding_ratio)
    pad_y = int(h * padding_ratio)

    H, W, _ = image.shape
    y1 = max(0, y_min - pad_y)
    y2 = min(H, y_max + pad_y)
    x1 = max(0, x_min - pad_x)
    x2 = min(W, x_max + pad_x)

    return image[y1:y2, x1:x2]

# Define landmark indices according to dlib's 68-point specification
# Left eye: points 36-41, Right eye: points 42-47, Nose: points 27-35, Mouth: points 48-67
regions = {
    "Full Face": image_rgb[rects[0].top():rects[0].bottom(), rects[0].left():rects[0].right()],
    "Left Eye": get_roi_crop(image_rgb, landmarks[36:42], padding_ratio=0.35),
    "Right Eye": get_roi_crop(image_rgb, landmarks[42:48], padding_ratio=0.35),
    "Nose": get_roi_crop(image_rgb, landmarks[27:36], padding_ratio=0.15),
    "Mouth": get_roi_crop(image_rgb, landmarks[48:68], padding_ratio=0.25)
}

# Plot the automatically extracted ROIs
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (name, crop) in zip(axes, regions.items()):
    ax.imshow(crop)
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Extract and save each model's feature maps

In [ ]:

import time
import pandas as pd
import numpy as np
import torch
from torchvision import transforms
from IPython.display import display as ipy_display

# --------------------------------------------------
# 1. IMAGE PREPROCESSING
# --------------------------------------------------

transform_imagenet = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Use explicit, reproducible input preparation.
def prepare_input(crop):
    return transform_imagenet(crop).unsqueeze(0).to(DEVICE)


# --------------------------------------------------
# 2. FEATURE EXTRACTION
# --------------------------------------------------

all_maps = {}
rows = []

@torch.inference_mode()
def extract_map(model_name, model, tensor, forward_fn):
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    fmap = forward_fn(model, tensor)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    fmap = fmap.detach().float().cpu()

    if not torch.isfinite(fmap).all():
        raise ValueError(
            f"Non-finite activations from {model_name}"
        )

    if fmap.ndim != 4:
        raise ValueError(
            f"{model_name} returned shape {tuple(fmap.shape)}; "
            "expected [batch, channels, height, width]."
        )

    return fmap, elapsed


# --------------------------------------------------
# 3. MODEL FORWARD FUNCTIONS
# --------------------------------------------------

def custom_forward(model, tensor):
    return model(tensor)


def resnet_forward(model, tensor):
    # ResNet-50 layer4 output, before global average pooling.
    x = model.conv1(tensor)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)
    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)
    x = model.layer4(x)
    return x


def vggface_forward(model, tensor):
    # VGGFace2 ResNet-50 final convolutional feature map.
    x = model.conv1(tensor)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)
    x = model.layer1(x)
    x = model.layer2(x)
    x = model.layer3(x)
    x = model.layer4(x)
    return x


# --------------------------------------------------
# 4. MODEL CONFIGURATION
# --------------------------------------------------

model_configs = [
    ("Custom CNN", model_custom, custom_forward),
    ("ResNet-50 ImageNet", model_imagenet, resnet_forward),
    ("ResNet-50 VGGFace2", model_vggface2, vggface_forward),
]


# --------------------------------------------------
# 5. WARM-UP
# --------------------------------------------------

for model_name, model, forward_fn in model_configs:
    sample = prepare_input(regions["Full Face"])

    try:
        model.eval()

        with torch.inference_mode():
            forward_fn(model, sample)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

    except Exception as e:
        raise RuntimeError(
            f"Warm-up failed for {model_name}: {e}"
        ) from e


# --------------------------------------------------
# 6. EXTRACT MAPS FOR ALL FACIAL REGIONS
# --------------------------------------------------

# Reset results so rerunning this cell won't duplicate rows.
all_maps.clear()
rows.clear()

for model_name, model, forward_fn in model_configs:
    print(f"\nExtracting features: {model_name}")

    for region_name, crop in regions.items():
        tensor = prepare_input(crop)

        fmap, elapsed = extract_map(
            model_name,
            model,
            tensor,
            forward_fn
        )

        key = (model_name, region_name)
        all_maps[key] = fmap.clone()

        arr = fmap.numpy()

        rows.append({
            "Model": model_name,
            "Region": region_name,
            "Feature Map Shape": str(tuple(fmap.shape)),
            "Channels": int(fmap.shape[1]),
            "Height": int(fmap.shape[2]),
            "Width": int(fmap.shape[3]),
            "Runtime (seconds)": float(elapsed),
            "Min": float(arr.min()),
            "Max": float(arr.max()),
            "Mean": float(arr.mean()),
            "Std": float(arr.std()),
            "Zero Fraction": float(np.mean(arr == 0)),
            "Finite Fraction": float(np.isfinite(arr).mean()),
            "Embedding L2 Norm": float(np.linalg.norm(arr)),
        })

        print(
            f"  {region_name}: "
            f"shape={tuple(fmap.shape)}, "
            f"time={elapsed:.4f}s"
        )


# --------------------------------------------------
# 7. DISPLAY AND SAVE RESULTS
# --------------------------------------------------

results_df = pd.DataFrame(rows)

# Use ipy_display instead of display because the
# notebook's display name may have been overwritten.
ipy_display(results_df.round(5))

output_path = "/content/feature_extraction_results.csv"
results_df.to_csv(output_path, index=False)

print(f"\nSaved results to: {output_path}")
print(f"Total records: {len(results_df)}")
print(f"Total feature maps stored: {len(all_maps)}")


## Visualise all three models correctly

In [ ]:
def display_channel(channel):
    channel = np.asarray(channel, dtype=np.float32)
    lo, hi = np.percentile(channel, [1, 99])

    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-8:
        return np.zeros_like(channel)

    return np.clip((channel - lo) / (hi - lo), 0, 1)

def plot_model_comparison(region_name="Full Face", num_channels=8):
    fig, axes = plt.subplots(
        len(model_configs), num_channels,
        figsize=(2.2 * num_channels, 7)
    )

    for row_idx, (model_name, _, _) in enumerate(model_configs):
        fmap = all_maps[(model_name, region_name)][0].numpy()

        # Select channels with the largest spatial standard deviation
        channel_std = fmap.std(axis=(1, 2))
        selected = np.argsort(channel_std)[-num_channels:][::-1]

        print(
            f"{model_name} | {region_name} | shape={fmap.shape} | "
            f"min={fmap.min():.5f} | max={fmap.max():.5f} | "
            f"mean={fmap.mean():.5f} | std={fmap.std():.5f} | "
            f"zero fraction={np.mean(fmap == 0):.2%}"
        )

        for col_idx, channel_idx in enumerate(selected):
            ax = axes[row_idx, col_idx]
            ax.imshow(
                display_channel(fmap[channel_idx]),
                cmap="viridis", vmin=0, vmax=1
            )
            ax.set_title(f"Ch {channel_idx}", fontsize=8)
            ax.axis("off")

        axes[row_idx, 0].set_ylabel(model_name, fontsize=9)

    fig.suptitle(
        f"{region_name}: strongest spatial-variation channels",
        fontsize=13
    )
    plt.tight_layout()
    plt.show()

plot_model_comparison("Full Face")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def normalize_channel(img):
    img = np.asarray(img, dtype=np.float32)
    lo, hi = np.percentile(img, [1, 99])
    if hi - lo < 1e-8:
        return np.zeros_like(img)
    return np.clip((img - lo) / (hi - lo), 0, 1)

@torch.inference_mode()
def extract_stages_custom(model, tensor):
    # Extract progressive stages from the Custom CNN
    stages = {}
    x = tensor
    # Stage 1: Conv1 + ReLU
    x = model.features[0](x)
    x = model.features[1](x)
    stages["Stage 1"] = x.cpu()[0].numpy()

    # Stage 2: MaxPool + Conv2 + ReLU
    x = model.features[2](x)
    x = model.features[3](x)
    x = model.features[4](x)
    stages["Stage 2"] = x.cpu()[0].numpy()

    # Stage 3: MaxPool + Conv3 + ReLU
    x = model.features[5](x)
    x = model.features[6](x)
    x = model.features[7](x)
    stages["Stage 3"] = x.cpu()[0].numpy()
    return stages

@torch.inference_mode()
def extract_stages_resnet(model, tensor):
    # Extract progressive stages from ResNet-50
    stages = {}
    x = model.conv1(tensor)
    x = model.bn1(x)
    x = model.relu(x)
    stages["Stage 1 (conv1)"] = x.cpu()[0].numpy()

    x = model.maxpool(x)
    x = model.layer1(x)
    stages["Stage 2 (layer1)"] = x.cpu()[0].numpy()

    x = model.layer2(x)
    stages["Stage 3 (layer2)"] = x.cpu()[0].numpy()

    x = model.layer3(x)
    stages["Stage 4 (layer3)"] = x.cpu()[0].numpy()

    x = model.layer4(x)
    stages["Stage 5 (layer4)"] = x.cpu()[0].numpy()
    return stages

tensor_input = prepare_input(regions["Full Face"]).to(DEVICE)

# Extract progressive features for all three models
custom_stages = extract_stages_custom(model_custom, tensor_input)
imagenet_stages = extract_stages_resnet(model_imagenet, tensor_input)
vggface_stages = extract_stages_resnet(model_vggface2, tensor_input)

models_stages = [
    ("Custom CNN (Untrained)", custom_stages),
    ("ResNet-50 (ImageNet)", imagenet_stages),
    ("ResNet-50 (VGGFace2)", vggface_stages)
]

# Plot progressive representations side-by-side
num_channels_to_show = 4
for model_title, stages in models_stages:
    num_stages = len(stages)
    fig, axes = plt.subplots(num_stages, num_channels_to_show, figsize=(12, 2.2 * num_stages))

    for row_idx, (stage_name, fmap) in enumerate(stages.items()):
        # Choose top channels based on spatial variance
        channel_std = fmap.std(axis=(1, 2))
        selected_channels = np.argsort(channel_std)[-num_channels_to_show:][::-1]

        for col_idx, ch_idx in enumerate(selected_channels):
            ax = axes[row_idx, col_idx] if num_stages > 1 else axes[col_idx]
            normalized_ch = normalize_channel(fmap[ch_idx])

            ax.imshow(normalized_ch, cmap="viridis")
            ax.axis("off")
            if col_idx == 0:
                ax.set_title(f"{stage_name}\nCh {ch_idx}", fontsize=9, loc='left', color='darkblue', weight='bold')
            else:
                ax.set_title(f"Ch {ch_idx}", fontsize=8)

    fig.suptitle(f"Progressive Representations - {model_title}", fontsize=12, weight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_model_comparison("Left Eye")
plot_model_comparison("Right Eye")
plot_model_comparison("Nose")
plot_model_comparison("Mouth")

In [ ]:
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

# Prepare lists of regions
region_names = ["Full Face", "Left Eye", "Right Eye", "Nose", "Mouth"]

# Create a figure to display similarity matrices for each model
fig, axes = plt.subplots(1, len(model_configs), figsize=(18, 5))

for idx, (model_name, _, _) in enumerate(model_configs):
    # Extract flattened features for all regions
    features_list = []
    for region in region_names:
        # Flatten the 3D/4D feature map into a 1D vector representing the region
        fmap_tensor = all_maps[(model_name, region)]
        flat_vector = fmap_tensor.view(-1).numpy().reshape(1, -1)
        features_list.append(flat_vector)

    # Stack and calculate Cosine Similarity between all regions
    features_stacked = np.vstack(features_list)
    similarity_matrix = cosine_similarity(features_stacked)

    # Plot similarity heatmaps
    ax = axes[idx]
    sns.heatmap(
        similarity_matrix,
        annot=True,
        fmt=".3f",
        xticklabels=region_names,
        yticklabels=region_names,
        cmap="coolwarm",
        vmin=0.0,
        vmax=1.0,
        ax=ax,
        cbar=(idx == len(model_configs) - 1)
    )
    ax.set_title(f"{model_name}\nCosine Similarity Map", fontsize=11, weight='bold')
    ax.set_xticklabels(region_names, rotation=45, ha='right')

plt.suptitle("Representational Similarity Analysis (RSA) of ROI Feature Extractions", fontsize=14, weight='bold', y=1.05)
plt.tight_layout()
plt.show()

### Progressive Layer-by-Layer Feature Map Visualization
The following cell hooks into different stages of the **ResNet-50 VGGFace2** model to capture, normalize, and compare how a single region (e.g., "Full Face") transforms progressively from low-level textures to high-level facial representations.

In [ ]:
from google.colab import files

files.download("/content/feature_extraction_results.csv")